In [32]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import math

In [ ]:
df = pd.read_csv("Coursera.csv")

df = df.head(700)

required_columns = [
    "course",
    "skills",
    "level",
    "partner"
    ]

for col in required_columns:
    if col not in df.columns:
        print(f"Missing column: {col}")

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna("")
    else:
        df[col] = df[col].fillna(0)

df.head()

In [34]:
df["combined_features"] = (
    df["course"].astype(str) + " " +
    df["skills"].astype(str) + " " +
    df["level"].astype(str) + " " +
    df["partner"].astype(str)
)

In [35]:
tfidf = TfidfVectorizer(stop_words="english")

matrix = tfidf.fit_transform(df["combined_features"])

similarity = cosine_similarity(matrix)

In [36]:
def recommend(course_name, top_n=10):

    course_name = course_name.strip().lower()

    matches = df[
        df["course"].str.lower().str.strip()
        == course_name
    ]

    if matches.empty:
        print("Course Not Found!")
        return None, []

    index = matches.index[0]

    scores = list(enumerate(similarity[index]))

    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = []

    for i in scores[1:top_n+1]:
        recommendations.append(i[0])

    return index, recommendations

In [37]:
def evaluate_metrics(relevance, total_relevant):

    k = len(relevance)

    precision = sum(relevance) / k

    recall = sum(relevance) / total_relevant

    if precision + recall == 0:
        f1 = 0
    else:
        f1 = (2 * precision * recall) / (precision + recall)

    hits = 0
    ap_sum = 0

    for rank, rel in enumerate(relevance, start=1):
        if rel == 1:
            hits += 1
            ap_sum += hits / rank

    map_score = ap_sum / hits if hits > 0 else 0

    dcg = 0

    for rank, rel in enumerate(relevance, start=1):
        dcg += rel / math.log2(rank + 1)

    ideal = sorted(relevance, reverse=True)

    idcg = 0

    for rank, rel in enumerate(ideal, start=1):
        idcg += rel / math.log2(rank + 1)

    ndcg = dcg / idcg if idcg > 0 else 0

    return {
        "Precision@10": round(precision, 3),
        "Recall@10": round(recall, 3),
        "F1-Score@10": round(f1, 3),
        "MAP@10": round(map_score, 3),
        "NDCG@10": round(ndcg, 3)
    }

In [38]:
course = input("Enter Course Name: ")



In [ ]:
target_index, recommendations = recommend(course)

if target_index is None:
    print("Course not found. Please enter a valid course name.")

else:

    print("\nRecommended Courses:\n")

    for i, idx in enumerate(recommendations, start=1):
        print(f"{i}. {df.iloc[idx]['course']}")

In [ ]:
if target_index is not None:

    target_skills = set(
        str(df.iloc[target_index]["skills"]).lower().split(",")
    )

    relevance = []

    for idx in recommendations:

        rec_skills = set(
            str(df.iloc[idx]["skills"]).lower().split(",")
        )

        intersection = len(target_skills & rec_skills)
        union = len(target_skills | rec_skills)

        jaccard = intersection / union if union > 0 else 0

        if jaccard >= 0.20:
            relevance.append(1)
        else:
            relevance.append(0)

    print("Relevance:", relevance)

In [ ]:
total_relevant = max(sum(relevance), 1)

metrics = evaluate_metrics(
    relevance,
    total_relevant
)

print( "\nEvaluation Metrics\n")
    

for key, value in metrics.items():

        print(key,":",value)